# Algorand Blockchain Behavior Risk Assessment Demo

This notebook demonstrates the comprehensive blockchain behavior risk assessment system
for Algorand DeFi lending protocols.

## Features Demonstrated:
- Transaction pattern analysis
- MEV exploitation detection
- Wallet clustering and Sybil attack detection
- Cross-chain bridge risk assessment
- Anomaly detection
- Comprehensive risk scoring
- Alert generation

In [ ]:
# Import required libraries
import sys
import os
sys.path.append('..')

import asyncio
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import json
from typing import Dict, List, Any

# Import our risk assessment modules
from common.models.blockchain_risk import (
    TransactionPattern, BehaviorPattern, RiskLevel, HolisticRiskProfile
)
from common.algorand.transaction_analyzer import AlgorandTransactionAnalyzer
from common.algorand.wallet_clustering import WalletClusteringAnalyzer
from common.algorand.mev_detector import MEVDetector
from common.utils.anomaly_detection import AnomalyDetector
from common.utils.risk_scoring import BlockchainRiskScorer, ScoringContext

from core.behavior_engine import BlockchainBehaviorEngine
from core.transaction_risk import TransactionRiskAnalyzer

# Set up plotting
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
%matplotlib inline

## 1. Generate Sample Data

Let's create realistic sample data representing different types of blockchain behaviors.

In [ ]:
class DemoDataGenerator:
    """Generate demonstration data for risk assessment"""
    
    @staticmethod
    def generate_normal_wallet_activity(wallet_id: str, days: int = 30) -> List[Dict[str, Any]]:
        """Generate normal wallet activity"""
        transactions = []
        base_time = datetime.utcnow() - timedelta(days=days)
        
        # Regular daily transactions with some randomness
        for day in range(days):
            daily_txn_count = np.random.poisson(3)  # Average 3 transactions per day
            
            for txn in range(daily_txn_count):
                timestamp = base_time + timedelta(
                    days=day,
                    hours=np.random.randint(8, 20),  # Active during day
                    minutes=np.random.randint(0, 60)
                )
                
                transactions.append({
                    'id': f'{wallet_id}_txn_{day}_{txn}',
                    'timestamp': timestamp.isoformat(),
                    'sender': wallet_id,
                    'receiver': f'merchant_{np.random.randint(1, 10)}',
                    'amount': np.random.lognormal(4, 1),  # Log-normal distribution
                    'asset-id': 0,
                    'tx-type': 'pay',
                    'fee': 0.001,
                    'confirmed-round': 1000000 + day * 100 + txn
                })
        
        return transactions
    
    @staticmethod
    def generate_suspicious_velocity_pattern(wallet_id: str) -> List[Dict[str, Any]]:
        """Generate suspicious high-velocity trading pattern"""
        transactions = []
        base_time = datetime.utcnow() - timedelta(hours=2)
        
        # Burst of 50 transactions in 2 hours
        for i in range(50):
            timestamp = base_time + timedelta(minutes=i * 2.4)  # Every ~2.4 minutes
            
            transactions.append({
                'id': f'{wallet_id}_burst_{i}',
                'timestamp': timestamp.isoformat(),
                'sender': wallet_id,
                'receiver': f'target_{i % 3}',  # Limited counterparties
                'amount': 1000,  # Identical amounts
                'asset-id': 100,
                'tx-type': 'axfer',
                'fee': 0.001,
                'confirmed-round': 2000000 + i
            })
        
        return transactions
    
    @staticmethod
    def generate_mev_sandwich_attack() -> List[Dict[str, Any]]:
        """Generate MEV sandwich attack pattern"""
        base_time = datetime.utcnow() - timedelta(minutes=30)
        block_number = 3000000
        
        return [
            {
                'id': 'mev_front_run',
                'timestamp': base_time.isoformat(),
                'sender': 'mev_bot_wallet',
                'receiver': 'tinyman_amm',
                'amount': 50000,
                'asset-id': 200,
                'tx-type': 'appl',
                'application-id': 552635992,  # Tinyman AMM
                'fee': 0.002,  # Higher fee for priority
                'confirmed-round': block_number,
                'intra-round-offset': 0
            },
            {
                'id': 'victim_transaction',
                'timestamp': base_time.isoformat(),
                'sender': 'innocent_user',
                'receiver': 'tinyman_amm',
                'amount': 25000,
                'asset-id': 200,
                'tx-type': 'appl',
                'application-id': 552635992,
                'fee': 0.001,
                'confirmed-round': block_number,
                'intra-round-offset': 1
            },
            {
                'id': 'mev_back_run',
                'timestamp': base_time.isoformat(),
                'sender': 'mev_bot_wallet',
                'receiver': 'tinyman_amm',
                'amount': 48500,  # Profit of 1500
                'asset-id': 200,
                'tx-type': 'appl',
                'application-id': 552635992,
                'fee': 0.002,
                'confirmed-round': block_number,
                'intra-round-offset': 2
            }
        ]
    
    @staticmethod
    def generate_sybil_cluster() -> Dict[str, List[Dict[str, Any]]]:
        """Generate Sybil attack cluster"""
        cluster_data = {}
        funding_wallet = 'funding_source'
        base_time = datetime.utcnow() - timedelta(days=7)
        
        # Create 5 related wallets
        for i in range(5):
            wallet_id = f'sybil_wallet_{i}'
            transactions = []
            
            # Initial funding (coordinated timing)
            funding_time = base_time + timedelta(minutes=i * 5)
            transactions.append({
                'id': f'funding_{i}',
                'timestamp': funding_time.isoformat(),
                'sender': funding_wallet,
                'receiver': wallet_id,
                'amount': 10000,
                'asset-id': 0,
                'tx-type': 'pay'
            })
            
            # Similar behavioral patterns
            for day in range(7):
                # Each wallet trades at similar times with similar amounts
                trade_time = base_time + timedelta(days=day, hours=10, minutes=i * 2)
                transactions.append({
                    'id': f'sybil_trade_{i}_{day}',
                    'timestamp': trade_time.isoformat(),
                    'sender': wallet_id,
                    'receiver': 'target_exchange',
                    'amount': 500,  # Same amounts
                    'asset-id': 100,
                    'tx-type': 'axfer'
                })
            
            cluster_data[wallet_id] = transactions
        
        return cluster_data

# Generate demo data
print("Generating demonstration data...")

# Normal user
normal_user_txns = DemoDataGenerator.generate_normal_wallet_activity('normal_user_wallet')

# Suspicious velocity user
velocity_user_txns = DemoDataGenerator.generate_suspicious_velocity_pattern('velocity_trader')

# MEV bot
mev_txns = DemoDataGenerator.generate_mev_sandwich_attack()

# Sybil cluster
sybil_cluster = DemoDataGenerator.generate_sybil_cluster()

print(f"Generated:")
print(f"- Normal user: {len(normal_user_txns)} transactions")
print(f"- Velocity trader: {len(velocity_user_txns)} transactions")
print(f"- MEV attack: {len(mev_txns)} transactions")
print(f"- Sybil cluster: {len(sybil_cluster)} wallets")

## 2. Transaction Pattern Analysis

Let's analyze transaction patterns to detect suspicious behaviors.

In [ ]:
# Initialize analyzers
transaction_analyzer = AlgorandTransactionAnalyzer()
anomaly_detector = AnomalyDetector()

async def analyze_transaction_patterns():
    """Analyze transaction patterns for different wallet types"""
    results = {}
    
    # Analyze normal user
    print("Analyzing normal user patterns...")
    normal_patterns = await transaction_analyzer.analyze_wallet_transactions(
        'normal_user_wallet', normal_user_txns, timedelta(days=30)
    )
    results['normal_user'] = {
        'patterns': normal_patterns,
        'risk_score': transaction_analyzer.calculate_pattern_risk_score(normal_patterns)
    }
    
    # Analyze velocity trader
    print("Analyzing velocity trader patterns...")
    velocity_patterns = await transaction_analyzer.analyze_wallet_transactions(
        'velocity_trader', velocity_user_txns, timedelta(days=1)
    )
    results['velocity_trader'] = {
        'patterns': velocity_patterns,
        'risk_score': transaction_analyzer.calculate_pattern_risk_score(velocity_patterns)
    }
    
    return results

# Run analysis
pattern_results = await analyze_transaction_patterns()

# Display results
for wallet_type, data in pattern_results.items():
    print(f"\n{wallet_type.upper()} ANALYSIS:")
    print(f"Risk Score: {data['risk_score']:.3f}")
    print(f"Patterns Detected: {len(data['patterns'])}")
    
    for pattern in data['patterns']:
        print(f"  - {pattern.pattern_type.value}: confidence {pattern.confidence_score:.2f}")
        print(f"    Anomalies: {', '.join(pattern.anomaly_indicators[:2])}")

## 3. MEV Attack Detection

Let's detect MEV exploitation patterns in transaction data.

In [ ]:
from common.algorand.mev_detector import MEVTransaction, MEVType

# Convert MEV transaction data to proper format
def convert_to_mev_transactions(txn_data):
    mev_txns = []
    for txn in txn_data:
        mev_txn = MEVTransaction(
            txn_id=txn['id'],
            block_number=txn['confirmed-round'],
            position_in_block=txn.get('intra-round-offset', 0),
            timestamp=datetime.fromisoformat(txn['timestamp']),
            sender=txn['sender'],
            receiver=txn['receiver'],
            amount=txn['amount'],
            asset_id=txn['asset-id'],
            application_id=txn.get('application-id'),
            fee=txn['fee'],
            gas_used=None,
            transaction_type=txn['tx-type']
        )
        mev_txns.append(mev_txn)
    return mev_txns

# Initialize MEV detector
mev_detector = MEVDetector()

async def detect_mev_patterns():
    """Detect MEV patterns in transaction data"""
    
    # Convert transaction data
    mev_transactions = convert_to_mev_transactions(mev_txns)
    
    # Detect MEV patterns
    print("Detecting MEV patterns...")
    patterns = await mev_detector.detect_mev_patterns(mev_transactions)
    
    # Generate analytics
    analytics = mev_detector.generate_mev_analytics(patterns)
    
    return patterns, analytics

# Run MEV detection
mev_patterns, mev_analytics = await detect_mev_patterns()

print("MEV DETECTION RESULTS:")
print(f"Total MEV Patterns Detected: {len(mev_patterns)}")
print(f"Total Extracted Value: {mev_analytics.total_mev_extracted:.2f}")

for pattern in mev_patterns:
    print(f"\nPattern: {pattern.mev_type.value}")
    print(f"Confidence: {pattern.confidence_score:.2f}")
    print(f"Extracted Value: {pattern.extracted_value:.2f}")
    print(f"Exploiter: {pattern.exploiter_address}")
    print(f"Victims: {', '.join(pattern.victim_addresses)}")

## 4. Wallet Clustering Analysis

Let's analyze wallet relationships to detect Sybil attacks.

In [ ]:
# Initialize clustering analyzer
clustering_analyzer = WalletClusteringAnalyzer()

async def analyze_wallet_clustering():
    """Analyze wallet clustering patterns"""
    
    print("Analyzing wallet clustering...")
    clusters = await clustering_analyzer.analyze_wallet_relationships(sybil_cluster)
    
    return clusters

# Run clustering analysis
detected_clusters = await analyze_wallet_clustering()

print("WALLET CLUSTERING RESULTS:")
print(f"Clusters Detected: {len(detected_clusters)}")

for cluster in detected_clusters:
    print(f"\nCluster ID: {cluster.cluster_id}")
    print(f"Wallet Count: {len(cluster.wallet_addresses)}")
    print(f"Cluster Score: {cluster.cluster_score:.3f}")
    print(f"Connection Strength: {cluster.connection_strength:.3f}")
    print(f"Shared Behaviors: {', '.join(cluster.shared_behaviors)}")
    print(f"Risk Indicators: {', '.join(cluster.risk_indicators)}")
    
    # Create Sybil alert if high risk
    if cluster.cluster_score > 0.7:
        alert = clustering_analyzer.create_sybil_alert(cluster)
        print(f"ALERT GENERATED: {alert.title} (Severity: {alert.severity.value})")

## 5. Risk Scoring and Assessment

Let's calculate comprehensive risk scores for different entities.

In [ ]:
# Initialize risk scorer
risk_scorer = BlockchainRiskScorer()

def create_risk_profile(entity_id, entity_type, patterns, anomalies=None, clusters=None):
    """Create a risk profile for scoring"""
    if anomalies is None:
        anomalies = []
    if clusters is None:
        clusters = []
    
    # Calculate risk factor scores
    risk_factor_scores = {
        'transaction_behavior': min(sum(p.confidence_score for p in patterns) / max(len(patterns), 1), 1.0),
        'anomaly_risk': max([a.score for a in anomalies], default=0.0),
        'clustering_risk': max([c.cluster_score for c in clusters], default=0.0),
        'mev_risk': 0.1,  # Base MEV risk
        'bridge_risk': 0.0
    }
    
    risk_factor_weights = {
        'transaction_behavior': 0.3,
        'anomaly_risk': 0.25,
        'clustering_risk': 0.25,
        'mev_risk': 0.15,
        'bridge_risk': 0.05
    }
    
    return HolisticRiskProfile(
        entity_id=entity_id,
        entity_type=entity_type,
        assessment_timestamp=datetime.utcnow(),
        transaction_patterns=patterns,
        defi_exposures=[],
        smart_contract_risks=[],
        liquidity_risks=[],
        governance_risks=[],
        asa_risks=[],
        protocol_risks=[],
        overall_risk_score=0.0,
        risk_level=RiskLevel.LOW,
        confidence_score=0.8,
        risk_factor_scores=risk_factor_scores,
        risk_factor_weights=risk_factor_weights,
        active_alerts=[],
        recommended_mitigations=[],
        data_completeness=0.9
    )

async def calculate_risk_scores():
    """Calculate risk scores for different entities"""
    
    scoring_context = ScoringContext(
        entity_type='wallet',
        time_horizon=timedelta(days=30),
        market_conditions='normal',
        regulatory_environment='moderate',
        protocol_maturity='established',
        analysis_purpose='lending'
    )
    
    results = {}
    
    # Score normal user
    normal_profile = create_risk_profile(
        'normal_user_wallet', 'wallet', 
        pattern_results['normal_user']['patterns']
    )
    
    normal_score, normal_level, normal_components = await risk_scorer.calculate_comprehensive_risk_score(
        normal_profile, scoring_context
    )
    
    results['normal_user'] = {
        'score': normal_score,
        'level': normal_level,
        'components': normal_components
    }
    
    # Score velocity trader
    velocity_profile = create_risk_profile(
        'velocity_trader', 'wallet',
        pattern_results['velocity_trader']['patterns']
    )
    
    velocity_score, velocity_level, velocity_components = await risk_scorer.calculate_comprehensive_risk_score(
        velocity_profile, scoring_context
    )
    
    results['velocity_trader'] = {
        'score': velocity_score,
        'level': velocity_level,
        'components': velocity_components
    }
    
    # Score Sybil cluster (if detected)
    if detected_clusters:
        sybil_profile = create_risk_profile(
            'sybil_cluster', 'wallet',
            [], [], detected_clusters
        )
        
        sybil_score, sybil_level, sybil_components = await risk_scorer.calculate_comprehensive_risk_score(
            sybil_profile, scoring_context
        )
        
        results['sybil_cluster'] = {
            'score': sybil_score,
            'level': sybil_level,
            'components': sybil_components
        }
    
    return results

# Calculate risk scores
risk_scores = await calculate_risk_scores()

print("COMPREHENSIVE RISK SCORES:")
print("=" * 50)

for entity, data in risk_scores.items():
    print(f"\n{entity.upper()}:")
    print(f"Overall Risk Score: {data['score']:.3f}")
    print(f"Risk Level: {data['level'].value}")
    print("Component Scores:")
    for component, score in data['components'].items():
        print(f"  - {component}: {score:.3f}")

## 6. Visualization

Let's create visualizations to better understand the risk assessment results.

In [ ]:
# Create risk score comparison chart
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Overall Risk Scores Comparison
entities = list(risk_scores.keys())
scores = [risk_scores[entity]['score'] for entity in entities]
levels = [risk_scores[entity]['level'].value for entity in entities]

colors = ['green', 'orange', 'red']
level_colors = [colors[0] if 'LOW' in level or 'MINIMAL' in level 
               else colors[1] if 'MODERATE' in level 
               else colors[2] for level in levels]

bars = axes[0,0].bar(entities, scores, color=level_colors, alpha=0.7)
axes[0,0].set_title('Overall Risk Scores by Entity')
axes[0,0].set_ylabel('Risk Score')
axes[0,0].set_ylim(0, 1)
axes[0,0].tick_params(axis='x', rotation=45)

# Add risk level labels
for bar, level in zip(bars, levels):
    height = bar.get_height()
    axes[0,0].text(bar.get_x() + bar.get_width()/2., height + 0.01,
                  level, ha='center', va='bottom', fontsize=8)

# 2. Component Risk Breakdown (for velocity trader)
if 'velocity_trader' in risk_scores:
    components = risk_scores['velocity_trader']['components']
    comp_names = list(components.keys())
    comp_scores = list(components.values())
    
    axes[0,1].barh(comp_names, comp_scores, color='skyblue', alpha=0.7)
    axes[0,1].set_title('Risk Component Breakdown - Velocity Trader')
    axes[0,1].set_xlabel('Risk Score')
    axes[0,1].set_xlim(0, 1)

# 3. Transaction Volume Over Time (Normal vs Velocity)
def plot_transaction_timeline(transactions, label, ax, color):
    times = [datetime.fromisoformat(txn['timestamp']) for txn in transactions]
    amounts = [txn['amount'] for txn in transactions]
    
    ax.scatter(times, amounts, alpha=0.6, label=label, color=color, s=20)

plot_transaction_timeline(normal_user_txns[-20:], 'Normal User', axes[1,0], 'green')
plot_transaction_timeline(velocity_user_txns[-20:], 'Velocity Trader', axes[1,0], 'red')

axes[1,0].set_title('Transaction Patterns Comparison')
axes[1,0].set_xlabel('Time')
axes[1,0].set_ylabel('Transaction Amount')
axes[1,0].legend()
axes[1,0].tick_params(axis='x', rotation=45)

# 4. MEV Detection Results
if mev_patterns:
    mev_types = [p.mev_type.value for p in mev_patterns]
    mev_values = [p.extracted_value for p in mev_patterns]
    mev_confidence = [p.confidence_score for p in mev_patterns]
    
    scatter = axes[1,1].scatter(mev_values, mev_confidence, 
                               c=range(len(mev_patterns)), 
                               cmap='viridis', s=100, alpha=0.7)
    axes[1,1].set_title('MEV Patterns Detected')
    axes[1,1].set_xlabel('Extracted Value')
    axes[1,1].set_ylabel('Confidence Score')
    
    # Add annotations
    for i, mev_type in enumerate(mev_types):
        axes[1,1].annotate(mev_type, (mev_values[i], mev_confidence[i]),
                          xytext=(5, 5), textcoords='offset points', fontsize=8)
else:
    axes[1,1].text(0.5, 0.5, 'No MEV Patterns\nDetected', 
                  ha='center', va='center', transform=axes[1,1].transAxes)
    axes[1,1].set_title('MEV Detection Results')

plt.tight_layout()
plt.show()

## 7. Risk Report Generation

Let's generate a comprehensive risk assessment report.

In [ ]:
def generate_risk_report():
    """Generate comprehensive risk assessment report"""
    
    report = {
        'report_timestamp': datetime.utcnow().isoformat(),
        'analysis_summary': {
            'entities_analyzed': len(risk_scores),
            'patterns_detected': sum(len(data['patterns']) for data in pattern_results.values()),
            'mev_patterns_detected': len(mev_patterns),
            'clusters_detected': len(detected_clusters)
        },
        'risk_assessment': risk_scores,
        'security_alerts': [],
        'recommendations': []
    }
    
    # Generate alerts for high-risk entities
    for entity, data in risk_scores.items():
        if data['score'] > 0.7:
            report['security_alerts'].append({
                'entity': entity,
                'alert_type': 'HIGH_RISK_SCORE',
                'severity': data['level'].value,
                'score': data['score'],
                'description': f'Entity {entity} has elevated risk score of {data["score"]:.3f}'
            })
    
    # Generate MEV alerts
    for pattern in mev_patterns:
        if pattern.confidence_score > 0.7:
            report['security_alerts'].append({
                'entity': pattern.exploiter_address,
                'alert_type': f'MEV_{pattern.mev_type.value.upper()}',
                'severity': 'HIGH',
                'confidence': pattern.confidence_score,
                'extracted_value': pattern.extracted_value,
                'description': f'MEV exploitation detected: {pattern.mev_type.value}'
            })
    
    # Generate recommendations
    if any(data['score'] > 0.5 for data in risk_scores.values()):
        report['recommendations'].extend([
            'Implement enhanced monitoring for high-risk entities',
            'Consider transaction limits for elevated risk accounts',
            'Review lending terms for moderate to high-risk borrowers'
        ])
    
    if mev_patterns:
        report['recommendations'].extend([
            'Implement MEV protection mechanisms',
            'Consider fair sequencing protocols',
            'Monitor for sandwich attack patterns'
        ])
    
    if detected_clusters:
        report['recommendations'].extend([
            'Investigate potential Sybil attack clusters',
            'Implement wallet relationship analysis',
            'Enhanced KYC for clustered addresses'
        ])
    
    return report

# Generate report
risk_report = generate_risk_report()

print("BLOCKCHAIN RISK ASSESSMENT REPORT")
print("=" * 60)
print(f"Generated: {risk_report['report_timestamp']}")
print(f"\nANALYSIS SUMMARY:")
for key, value in risk_report['analysis_summary'].items():
    print(f"  {key.replace('_', ' ').title()}: {value}")

print(f"\nSECURITY ALERTS ({len(risk_report['security_alerts'])}):")
for alert in risk_report['security_alerts']:
    print(f"  🚨 {alert['alert_type']} - {alert['entity']}")
    print(f"     Severity: {alert['severity']}")
    print(f"     {alert['description']}")
    print()

print(f"RECOMMENDATIONS ({len(risk_report['recommendations'])}):")
for i, rec in enumerate(risk_report['recommendations'], 1):
    print(f"  {i}. {rec}")

# Export report as JSON
with open('blockchain_risk_assessment_report.json', 'w') as f:
    json.dump(risk_report, f, indent=2, default=str)

print(f"\n📄 Report exported to: blockchain_risk_assessment_report.json")

## 8. Summary

This demonstration shows the comprehensive blockchain behavior risk assessment system for Algorand DeFi lending protocols. The system successfully:

### ✅ **Detects Suspicious Patterns:**
- Transaction velocity spikes
- Wash trading behaviors
- Dormant wallet activations
- MEV exploitation attacks
- Sybil attack clusters

### ✅ **Provides Risk Scoring:**
- Blockchain-native risk assessment
- Component-based scoring
- Context-aware adjustments
- Portfolio-level analysis

### ✅ **Generates Actionable Insights:**
- Real-time security alerts
- Risk mitigation recommendations
- Comprehensive reporting
- Visual risk analysis

### 🔧 **Key Features:**
- **Algorand-Native**: Built specifically for Algorand blockchain patterns
- **No Traditional Banking**: Pure blockchain risk assessment
- **Real-time Analysis**: Suitable for live transaction monitoring
- **Comprehensive Coverage**: Multi-dimensional risk analysis
- **Actionable Results**: Clear recommendations and alerts

This system replaces traditional credit scoring with blockchain behavior analysis, providing DeFi lending protocols with sophisticated risk assessment capabilities.